In [1]:
import torch
# torch.autograd.set_detect_anomaly(True)
from torch.utils.data import DataLoader,WeightedRandomSampler
from torch.optim.lr_scheduler import PolynomialLR
import torch.nn.functional as F
from torchvision.transforms import v2
import pandas as pd
import albumentations as A
# from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from pathlib import Path
from tqdm.notebook import tqdm
import json
import cv2
import matplotlib.pyplot as plt
import numpy as np
import random
import os
import zarr
%load_ext autoreload
%autoreload 2
###IE###
from utils.helpers import (
    read_images,load_model,bootstrap_model,generate_all_masks,
    make_result_dataloaders,make_training_dataloaders,
    draw_fast_plots
)
from utils.preprocessing import (
    normalize_xca,normalize_empty,get_preprocessing,
    remove_black_border_multithread,BrightnessMultiplicativeNNUNet2D,
    ContrastAugmentationNNUNet2D
)
# from utils.debuging import show_bootstraping_results
from models.nnunet import nnUnet
from models.memory_unet import MemoryUnet
from models.swin_encoder import SwinUperNet
from models.hrnet import HRNET
from models.moe_model import MoeUnet
from models.snake_unet import VAMoETED
from models.sarc_unet import SARC_UNet
from utils.testmodel import UTANet
from utils.losses import UnetLoss
from utils.recorder import HistoryRecorder
from logger import save_full_report
from trainer import trainer
from trainer import evaluation
from image_clusters import valid,problem_maps
from utils.graph_extraction import (
    create_gnn_dataset,generate_robust_mask,
    create_gnn_dataset_parallel_limited)
from models.Swin_Unet.networks.vision_transformer import  SwinUnet
from utils.debuging import weighting_logic , save_loss_weights , save_weight_overlay_cv2
###SS###

/home/parsa/Venvs/AI/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
args = {
    "paths":{
        "base_path" : "./dataset/syntax/",
        "root_path" : "./dataset/syntax/topology_dataset/labels/",
        "output_base_path" : "./outputs",
        "images_folder" : "origianl_preprocessed",
        "clahe_folder" : "origianl_preprocessed",
        "generated_folder" : "generated_mask_probs",
        "labels_folder" : "labels",
        "bootstrap_folder" : "bootstrap",
        "hard_vessel_folder" : "hard_vessels_mask",
        "filter_folder" : "filters",
        "bootstrap_seq":"./bootstrap_sequence",
        "topology_folder" : "topology_dataset",
        "side_folder":"./data/",
        "border_cords_folder" : "boarder_removed"
    },
    "memory_unet":{
        "layer_count":4,
        "channel_max":256,
        "downsampling":"maxpool",
        "dropout" : False,
        "normaliztion":"InstanceNorm2d",#InstanceNorm2d
        "activation" : "LeakyReLU",
        "first_layer_out_c" : 32,
    },
    "swin":{
        "encoder_out_channels":150
    },
    "moe_unet" : {
        "encoder_out_channels" : [[64,3],[128,4],[256,6],[512,3],[512,4]],
        "cbam_args":{"use_cbam":True,"ratio":16,"spatial_kernel":7},
        "resample_base_size" : None,
        "rp_out_channels" : 64,
        "router_counts":5,
        "expert_counts":3,
        "top_k":2,
        "train_moe":False,
        "aux_heads":False,
        "norm" : "BatchNorm2d"#InstanceNorm2d
    },
    "socre_params":{
        "p":0.4,
        "r":0.6,
        "choice_score" : "dice"
    },
    "image_shape" : (512,512),
    "in_c":1,
    "deep_super_vision":True,
    "bootstrap":False,
    "training_mode":"branch", # binary - multi - cleaning - branch
    "binary_training_mode" : "partial",# full - partial
    "report_class_wise":True,
    "attention" : False,
    "batch_size" : 6,
    "num_workers" : 2,
    "device" : "cuda" if torch.cuda.is_available() else "cpu",
    "epochs":100,
    "f_int_scale" : 2,
    "new_vessel_width":2,
    "full_report_cycle" : 10,
    "max_channels":512,
    "loss_type":"tversky",
    "name" : "branch-costum-nobias-cbam-upsample",
    "mask_ce_weights":None,
    "layer_count":5,
    "cut_borders":True,
    ### Loss Functions
    "ce_loss_conf":{
        "pos_weight":50.0,
        "bg_weight":1.0,
        "imp_coef":1.0
    },
    "tversky_conf":{
        "alpha":0.2,
        "beta":0.8,
        "gamma":1.00,
        "weights":None,
        "imp_coef":1.0
    },
    "aux_loss_conf":{
        "imp_coef" : 0.1
    },
    "cl_dice_conf":{
        "imp_coef":1.0,
        "k":40
    },
    ### Optimizers
    "sgd_conf":{
        "lr" : 0.1,
        "momentum" : 0.99,
        "weight_decay" : 3e-5,
        "nesterov":True
    },
    "adam_conf":{
        "lr" : 0.01,
        "weight_decay" : 3e-5,
    },
    "adamw_conf":{
        "lr" : 0.0001,
        "weight_decay" : 3e-5,
    }
}
if(args["training_mode"] == "binary" or args["training_mode"]=="branch"):
    args["class_count"] = 2 
    class_map = {
        1:"fg"
    }

else:
    args["class_count"] = 26
    class_map = {
        1: '1',2: '2', 3: '3',4: '4',
        5: '5',6: '6',7: '7',8: '8',
        9: '9',10: '9a',11: '10',12: '10a',
        13: '11',14: '12',15: '12a',16: '13',
        17: '14',18: '14a',19: '15',20: '16',
        21: '16a',22: '16b',23: '16c',
        24: '12b',25: '14b'
    }
hard_classes = [10,11,12,14,15,17,18,19,21,22,23,24,25]
# losses_keys = ["total loss","FCE loss",args["loss_type"]]
losses_keys = ["total loss","mask CE-Loss",args["loss_type"],"side_loss","aux_loss"]#,"side_loss"
if(args["deep_super_vision"]):
    shape = args["image_shape"]
    loss_weights = [1/(2**i) for i in range(1,len(args["moe_unet"]["encoder_out_channels"]))]
else:
    loss_weights = []
loss_weights



[0.5, 0.25, 0.125, 0.0625]

In [3]:
if(args["training_mode"]=="full"):
    b=0.999

    train_class_counts = [
        1200,374,375,369,303,525,537,
        198,340,70,21,310,1,61,320,129,
        63,305,107,49,127,38,232,43,48,31
    ]

    total = np.sum(train_class_counts)
    mask_ce_weights = np.log(total/np.array(train_class_counts))
    mask_ce_weights = (mask_ce_weights / mask_ce_weights.mean()).tolist()
    mask_ce_weights[0]=0.1

    args["mask_ce_weights"]=mask_ce_weights
    args["mask_ce_weights"]
elif(args["training_mode"]=="cleaning"):
    args["mask_ce_weights"] = [0.1,1]
else:
    # args["mask_ce_weights"]=[0.1,1]
    # args["tversky_conf"]["weights"] = [0.3,1]
    pass

In [4]:
train_transforms , test_transforms = get_preprocessing(args["training_mode"],args)

if(args["in_c"]==1):
    if(args["training_mode"]!="cleaning"):
        normalizer = normalize_xca
        print("set to z score")
    else:
        normalizer = normalize_empty
elif(args["in_c"]==2):
    print("set to z score 2c")
    normalizer = normalize_empty
else:
    normalizer = A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
        max_pixel_value=255.0,
    )
    print("set to imagenet")


set to z score


/home/parsa/Masters/Coronary Artry/Datasets/Arcade/codes/utils/preprocessing.py:207: UserWarning: Argument(s) 'alpha_affine' are not valid for transform ElasticTransform
  A.ElasticTransform(
/home/parsa/Masters/Coronary Artry/Datasets/Arcade/codes/utils/preprocessing.py:220: UserWarning: Argument(s) 'shift_limit' are not valid for transform OpticalDistortion
  A.OpticalDistortion(


In [5]:
train_images,sampler_weights,last_boot_level = read_images(
    paths = args["paths"],
    part = "train",
    chosen_labels=None,
    hard_classes = hard_classes,
    bootstrap = args["bootstrap"],
    training_mode = args["training_mode"],
    binary_mode = args["binary_training_mode"],
    in_c = args["in_c"],
    cut_borders = args["cut_borders"]

)
valid_images = read_images(
    paths = args["paths"],
    part = "val",
    chosen_labels=None,
    bootstrap = args["bootstrap"],
    training_mode = args["training_mode"],
    binary_mode = args["binary_training_mode"],
    in_c = args["in_c"],
    cut_borders = args["cut_borders"]
)
test_images = read_images(
    paths = args["paths"],
    part = "test",
    chosen_labels=None,
    bootstrap = args["bootstrap"],
    training_mode = args["training_mode"],
    binary_mode = args["binary_training_mode"],
    in_c = args["in_c"],
    cut_borders = args["cut_borders"]
)

0


  0%|          | 0/1000 [00:00<?, ?it/s]

0


  0%|          | 0/200 [00:00<?, ?it/s]

0


  0%|          | 0/300 [00:00<?, ?it/s]

In [6]:
# black_th=100
# min_vote=0.85
# remove_black_border_multithread(train_images,paths=args["paths"],part="train",
#                                 black_th=black_th,min_vote=min_vote,mode="plot")
# remove_black_border_multithread(valid_images,paths=args["paths"],part="val",
#                                 black_th=black_th,min_vote=min_vote,mode="plot")
# remove_black_border_multithread(test_images,paths=args["paths"],part="test",
#                                 black_th=black_th,min_vote=min_vote,mode="plot")

In [7]:
sampler = WeightedRandomSampler(
    weights=sampler_weights, 
    num_samples=len(sampler_weights),
    replacement=True
)
train_loader , valid_loader , test_loader = make_training_dataloaders(
    train_images=train_images,
    valid_images=valid_images,
    test_images=test_images,
    normalizer=normalizer,
    train_transforms=train_transforms,
    test_transforms=test_transforms,
    args=args,
    sampler=None
)
re_train_loader , re_valid_loader , re_test_loader  = make_result_dataloaders(
    train_images=train_images,
    valid_images=valid_images,
    test_images=test_images,
    normalizer=normalizer,
    transforms=test_transforms
)

# Make GNN Dataset

In [8]:
# generate_robust_mask(
#     model = best_model,
#     train_loader = re_train_loader,
#     valid_loader = re_valid_loader,
#     output_path = "./dataset/syntax/fixed_binary_masks",
#     low_thresh = 0.2,
#     high_thresh = 0.7,
#     save_mode = "plot"
# )

In [9]:
# model_name = "2026-06-01_12-12-57 [binary-tests-dsv-frangi+swin-boot_skel]"
# out_folder = f"./outputs/{model_name}/"
# best_model = load_model(
#     model_class=SwinUperNet,
#     path=out_folder,
#     model_inputs={"args":args},
#     device=args["device"]
# )
# dataloaders = {
#     "train":re_train_loader,
#     "val":re_valid_loader,
#     "test":re_test_loader
# }
# base_path = f"./dataset/syntax/gnn_dataset/{model_name}"

# create_gnn_dataset_parallel_limited(
#     model=best_model,
#     dataloaders=dataloaders,
#     args=args,
#     base_path=base_path,
#     low_thresh=0.2,
#     high_thresh=0.7,
#     enhance_mask=False,
#     paths=args["paths"],
#     labeling_mode="binary_kernel",
#     r=2, #1
#     min_votes=15, #5
#     save_pred_masks=True,
#     max_workers=4,
#     max_pending=8,
# )

# Bootstrap

In [10]:
# out_folders = [
#     "./outputs/2026-05-31_19-37-00 [binary-tests-dsv-frangi+swin-boot_skel]",
# ]
# dataloaders = {
#     "train":re_train_loader,
#     "val":re_valid_loader,
#     "test":re_test_loader,
# }
# last_boot_level = 3
# for out_folder in out_folders:
    
#     best_model = load_model(
#         model_class=SwinUperNet,
#         path=out_folder,
#         model_inputs={},
#         device=args["device"]
#     )
#     bootstrap_model(
#         model = best_model,
#         last_boot_level = last_boot_level,
#         args = args,
#         dataloaders = dataloaders,
#         gt_mask_mode = "pred_skel", #real_pred , pred_skel 
#         model_name = out_folder.split("/")[2],
#         iterations = 2,
#         new_vessel_width = args["new_vessel_width"],
#         comp = False,
#         use_erode = False
#     )
#     last_boot_level+=1
    

# Trained Model Outputs

In [11]:
# out_folder = "./outputs/2026-04-16_19-46-59 [binary-tests-dsv-clahe+sobel+swin]++/"
# best_model = load_model(
#     model_class=SwinUperNet,
#     path=out_folder,
#     model_inputs={"args":args},
#     device=args["device"]
# )

# df = model_performance_per_image(
#     model=best_model,
#     dataloader=re_valid_loader,
#     args=args,
#     out_folder="./data/model_performance",
#     valid_path="./data/valid_problem_df.csv"
# )
# generate_all_masks(
#     model=best_model,
#     train_loader=re_train_loader,
#     valid_loader=re_valid_loader,
#     test_loader=re_test_loader,
#     output_base="./dataset/syntax/"
# )

# generate_robust_mask(
#     model=best_model,
#     valid_loader=re_valid_loader,
#     train_loader=re_train_loader,
#     output_path="./dataset/syntax/fixed_binary_masks"
# )

In [12]:
import segmentation_models_pytorch as smp

class TEMP(torch.nn.Module):
    def __init__(self,args):
        super(TEMP,self).__init__()
        self.model = smp.DeepLabV3Plus(
            encoder_name="resnet50",
            encoder_weights="imagenet",
            in_channels=1,       
            classes=2,             
            activation=None,         
            encoder_output_stride=8,   
        )
        original_head = self.model.segmentation_head

        # 3. Wrap it in an nn.Sequential with a Spatial Dropout layer
        # A dropout probability (p) between 0.2 and 0.5 is standard
        self.model.segmentation_head = torch.nn.Sequential(
            torch.nn.Dropout2d(p=0.5),
            original_head
        )
    def _side_loss(self,s):
        return torch.tensor(0.0,device="cuda")
    def forward(self,x):
        if(self.training):
            return [self.model(x)],torch.tensor(0.0,device=x.device)
        else:
            return[self.model(x)]
    

# Train The Model

In [13]:
if(args["training_mode"]!="cleaning"):
    # model = SwinUperNet(args=args)
    # model = MemoryUnet(args=args)
    model = MoeUnet(args)
    # model = TEMP(args)
    # model = Tnet().cuda()
    # model.apply(init_resnet_style)
    # model = UTANet(pretrained=True, n_classes=2)
    
    model = model.to(args["device"])
    # optimizer = torch.optim.SGD(
    #     model.parameters(),
    #     **args["sgd_conf"]
    # )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        **args["adamw_conf"]
    )
else:
    model = MemoryUnet(args=args)
    model = model.to(args["device"])

    optimizer = torch.optim.SGD(
        model.parameters(),
        **args["sgd_conf"]
    )

loss_fn = UnetLoss(args)


lr_sch = PolynomialLR(optimizer=optimizer,total_iters=args["epochs"],power=0.9)
# lr_sch = None
recorder = HistoryRecorder(losses_keys=losses_keys,class_maps =class_map,
                           class_count=args["class_count"]-1)


# best_model = trainer(
#     args=args,
#     recorder = recorder,
#     model = model,
#     optimizer = optimizer,
#     loss_fn = loss_fn,
#     train_loader = train_loader,
#     valid_loader = valid_loader,
#     test_loader=test_loader,
#     loss_weights=loss_weights,
#     lr_sch = lr_sch,
#     choice_score= args["socre_params"]["choice_score"]
# )
# show_bootstraping_results(args)

[64, 3]
[128, 4]
[256, 6]
[512, 3]
[512, 4]
{0: {'in_c': 1, 'out_c': 64, 'conv_co': 0, 'do_downsample': False}, 1: {'in_c': 64, 'out_c': 64, 'conv_co': 3, 'do_downsample': False}, 2: {'in_c': 64, 'out_c': 128, 'conv_co': 4, 'do_downsample': True}, 3: {'in_c': 128, 'out_c': 256, 'conv_co': 6, 'do_downsample': True}, 4: {'in_c': 256, 'out_c': 512, 'conv_co': 3, 'do_downsample': True}, 5: {'in_c': 512, 'out_c': 512, 'conv_co': 4, 'do_downsample': True}}
{1: {'skip_c': 64, 'in_c': 64, 'out_c': 64}, 2: {'skip_c': 64, 'in_c': 128, 'out_c': 64}, 3: {'skip_c': 128, 'in_c': 256, 'out_c': 128}, 4: {'skip_c': 256, 'in_c': 512, 'out_c': 256}, 5: {'skip_c': 512, 'in_c': 512, 'out_c': 512}}
loss is set to tversky


# Test Evaluation

# Model Results

In [14]:
# dataloaders = {
#     "train":re_train_loader,
#     "val":re_valid_loader,
#     "test":re_test_loader,
# }
# input_boot_strap_path = Path("./dataset/syntax/bootstrap")
# output_results_path = Path("./bootstrap_sequence")

# steps = os.listdir(input_boot_strap_path)
# steps = list(map(int, steps))
# steps.sort()

# for part,dataloader in dataloaders.items():
#     out_path = output_results_path/part
#     os.makedirs(out_path,exist_ok=True)

#     for img_base, img, mask, side, name in tqdm(dataloader):
#         image = img_base.squeeze(0).permute(1,2,0).numpy().copy()
#         name = str(name[0])
#         gt_image = image.copy()
#         gt_image[mask.numpy()[0]!=0] = [0,0,255]
        
#         masks = [image,gt_image]
#         for step in steps : 
#             input_path = input_boot_strap_path / str(step) / part / f"{name}.png"
#             b_mask = cv2.imread(input_path,cv2.IMREAD_GRAYSCALE)
#             img = image.copy()
#             img[b_mask!=0] = [0,0,255]
#             masks.append(img)
#         full_plot = draw_fast_plots(masks,plot_shape=(2,3))
#         cv2.imwrite(out_path / f"{name}.png",full_plot)

In [16]:
model_name = "2026-06-22_22-37-02 [branch-costum-nobias-cbam-upsample]"
out_folder = f"./outputs/{model_name}"
best_model = load_model(
    model_class=TEMP,
    path=out_folder,
    model_inputs={},
    device=args["device"]
)
best_model.eval()
dataloaders = {
    "train":re_train_loader,
    "val":re_valid_loader,
    "test":re_test_loader,
}
out_base = Path("./dataset/syntax/topology_pred_dataset")
with torch.no_grad():
    for part , dl in dataloaders.items():
        os.makedirs(out_base/part,exist_ok=True)
        for data in tqdm(dl):
            img = data["new_image"].cuda()
            name =data["name"][0]
            with torch.autocast(device_type="cuda",dtype=torch.float16):
                pred_logits = best_model(img)[0]
                pred_class = torch.argmax(pred_logits,dim=1)
            pred_class = pred_class.cpu().numpy()[0]
            cv2.imwrite(
                out_base/part/f"{name}.png",
                (pred_class*255)
            )

  0%|          | 0/1000 [00:00<?, ?it/s]

[ WARN:0@39.072] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

# Save Results

In [ ]:
save_full_report(
    recorder= recorder , 
    output_base_path=args["paths"]["output_base_path"],
    model=best_model,
    valid_loader=valid_loader,
    normalizer = normalizer,
    args=args,
    class_map=class_map,
    name=args["name"],
    valid_images = valid_images,
    test_transforms = test_transforms,
    notebook_name = "Multi_Main.ipynb",
    class_count = args["class_count"],
    device = args["device"],
    training_mode = args["training_mode"],
    last_boot_level=last_boot_level
)